In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [0]:
import numpy as np
import mlflow
from mlflow import MlflowClient
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score

from churn.config import GOLD_TABLE, MODEL_NAME, ID, TARGET, RANDOM_SEED, EXPORTS_VOLUME
from churn.validation import validar_gold

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

In [0]:
import sys
sys.version

In [0]:
pdf = spark.table(GOLD_TABLE).toPandas()
validar_gold(pdf)

X = pdf.drop(columns=[ID, TARGET])
y = pdf[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

assert np.abs(y_train.mean() - y_test.mean()) < 0.01, "La proporción de churn en train y test no es la misma"

In [0]:
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
modelo = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/{champion.version}")

pr_auc_ahora = average_precision_score(y_test, modelo.predict(X_test)[:, 1])
pr_auc_registrada = client.get_run(champion.run_id).data.metrics["pr_auc_test"]

assert np.isclose(pr_auc_ahora, pr_auc_registrada), f"Split distinto: {pr_auc_ahora} frente a {pr_auc_registrada}"

In [0]:
gold_split = pdf.copy()
gold_split["conjunto"] = "train"
gold_split.loc[X_test.index, "conjunto"] = "test"

gold_split.to_csv(f"{EXPORTS_VOLUME}/gold_split.csv", index=False)